# Notebook 1 — EDA & Baseline Models
**Member 1 Responsibility**

This notebook covers:
1. Exploratory Data Analysis (EDA) of the AmbiStory dataset
2. Three baseline models: Random, Mean, TF-IDF
3. Evaluation on the dev set using official metrics

Run this notebook first. It saves `baselines_results.json` for the report notebook.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import random
import warnings
warnings.filterwarnings('ignore')

# ── Load data ──────────────────────────────────────────────────────────────
with open('train.json') as f:
    train_raw = json.load(f)
with open('dev.json') as f:
    dev_raw = json.load(f)

def to_df(raw):
    rows = []
    for sid, s in raw.items():
        rows.append({
            'id': sid,
            'homonym': s['homonym'],
            'judged_meaning': s['judged_meaning'],
            'precontext': s.get('precontext', ''),
            'sentence': s['sentence'],
            'ending': s.get('ending', '') or '',
            'average': s['average'],
            'stdev': s['stdev'],
            'choices': s['choices'],
            'example_sentence': s.get('example_sentence', '') or ''
        })
    return pd.DataFrame(rows)

train_df = to_df(train_raw)
dev_df   = to_df(dev_raw)
print(f'Train: {len(train_df)} samples | Dev: {len(dev_df)} samples')

## 1. Exploratory Data Analysis

In [ ]:
# ── Distribution of average plausibility scores ─────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Score distribution
axes[0].hist(train_df['average'], bins=20, color='steelblue', edgecolor='white', linewidth=0.5)
axes[0].set_title('Distribution of Average Plausibility Scores (Train)', fontsize=11)
axes[0].set_xlabel('Average Score (1–5)')
axes[0].set_ylabel('Count')
axes[0].axvline(train_df['average'].mean(), color='red', linestyle='--', label=f'Mean={train_df["average"].mean():.2f}')
axes[0].legend()

# Standard deviation distribution
axes[1].hist(train_df['stdev'], bins=20, color='coral', edgecolor='white', linewidth=0.5)
axes[1].set_title('Distribution of Annotation Std Dev (Train)', fontsize=11)
axes[1].set_xlabel('Std Dev')
axes[1].set_ylabel('Count')
axes[1].axvline(train_df['stdev'].mean(), color='navy', linestyle='--', label=f'Mean={train_df["stdev"].mean():.2f}')
axes[1].legend()

# Score category breakdown
bins = pd.cut(train_df['average'], bins=[0.9, 1.5, 2.5, 3.5, 4.5, 5.1],
              labels=['1', '1.5–2.5', '2.5–3.5', '3.5–4.5', '5'])
bins.value_counts().sort_index().plot(kind='bar', ax=axes[2], color='mediumpurple', edgecolor='white')
axes[2].set_title('Samples per Score Bucket (Train)', fontsize=11)
axes[2].set_xlabel('Score Bucket')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('eda_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_score_distribution.png')

In [ ]:
# ── Annotator disagreement analysis ─────────────────────────────────────
print('=== Dataset Statistics ===')
print(f'\nTrain average score: {train_df["average"].mean():.3f} ± {train_df["average"].std():.3f}')
print(f'Train stdev (annotation disagreement): {train_df["stdev"].mean():.3f}')
print(f'Dev average score:   {dev_df["average"].mean():.3f} ± {dev_df["average"].std():.3f}')

# How many unique homonyms?
print(f'\nUnique homonyms in train: {train_df["homonym"].nunique()}')
print(f'Unique homonyms in dev:   {dev_df["homonym"].nunique()}')

# How often does an ending exist?
has_ending_train = (train_df['ending'] != '').sum()
print(f'\nSamples with ending (train): {has_ending_train}/{len(train_df)} ({100*has_ending_train/len(train_df):.1f}%)')

# Top homonyms by frequency
print('\nTop 10 homonyms by frequency:')
print(train_df['homonym'].value_counts().head(10).to_string())

In [ ]:
# ── Correlation: stdev vs average score (annotator agreement by score) ──
fig, ax = plt.subplots(figsize=(7, 4))
sc = ax.scatter(train_df['average'], train_df['stdev'], alpha=0.3, s=12, c='steelblue')
ax.set_xlabel('Average Plausibility Score')
ax.set_ylabel('Annotation Std Dev')
ax.set_title('Annotator Disagreement vs. Score\n(higher stdev = more disagreement)')
plt.tight_layout()
plt.savefig('eda_stdev_vs_score.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: eda_stdev_vs_score.png')

## 2. Evaluation Helper Functions

In [ ]:
def evaluate(preds, df):
    """
    Compute both official metrics.
    preds: list of floats (predictions in same order as df rows)
    df:    DataFrame with 'average' and 'stdev' columns
    Returns dict with spearman_r, accuracy_within_stdev
    """
    preds = np.array(preds)
    trues = df['average'].values
    stdevs = df['stdev'].values

    # Spearman correlation
    rho, pval = spearmanr(preds, trues)

    # Accuracy within 1 std dev (std dev is at least 1 per task description)
    margin = np.maximum(stdevs, 1.0)
    within = np.abs(preds - trues) <= margin
    acc = within.mean()

    return {'spearman_r': round(float(rho), 4), 'acc_within_stdev': round(float(acc), 4)}

print('Evaluation function defined.')

## 3. Baseline 1 — Random Prediction

In [ ]:
random.seed(42)
random_preds = [random.randint(1, 5) for _ in range(len(dev_df))]
random_results = evaluate(random_preds, dev_df)
print('Random baseline:', random_results)

## 4. Baseline 2 — Global Mean Prediction

In [ ]:
global_mean = train_df['average'].mean()
mean_preds = [global_mean] * len(dev_df)
mean_results = evaluate(mean_preds, dev_df)
print(f'Global mean = {global_mean:.3f}')
print('Mean baseline:', mean_results)

## 5. Baseline 3 — TF-IDF Nearest Neighbour

In [ ]:
def build_context(row):
    """Concatenate story fields into a single string for TF-IDF."""
    return ' '.join([
        row['precontext'],
        row['sentence'],
        row['ending'],
        row['judged_meaning']
    ]).strip()

train_texts = train_df.apply(build_context, axis=1).tolist()
dev_texts   = dev_df.apply(build_context, axis=1).tolist()

# Fit TF-IDF on train, transform both
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
train_vecs = tfidf.fit_transform(train_texts)
dev_vecs   = tfidf.transform(dev_texts)

# For each dev sample, find most similar train sample and use its average score
# Process in batches to avoid memory issues
tfidf_preds = []
batch_size = 50
for i in range(0, len(dev_df), batch_size):
    sims = cosine_similarity(dev_vecs[i:i+batch_size], train_vecs)
    best_idxs = sims.argmax(axis=1)
    for idx in best_idxs:
        tfidf_preds.append(train_df.iloc[idx]['average'])

tfidf_results = evaluate(tfidf_preds, dev_df)
print('TF-IDF NN baseline:', tfidf_results)

## 6. Summary Table & Save Results

In [ ]:
results = {
    'Random':    random_results,
    'Mean':      mean_results,
    'TF-IDF NN': tfidf_results,
}

res_df = pd.DataFrame(results).T
res_df.index.name = 'Model'
print('\n=== Baseline Results on Dev Set ===')
print(res_df.to_string())

# ── Save for report notebook ─────────────────────────────────────────────
with open('baselines_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved: baselines_results.json')

In [ ]:
# ── Visual comparison ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
models = list(results.keys())
spearman_vals = [results[m]['spearman_r'] for m in models]
acc_vals      = [results[m]['acc_within_stdev'] for m in models]

colors = ['#e74c3c', '#3498db', '#2ecc71']
axes[0].bar(models, spearman_vals, color=colors, edgecolor='white')
axes[0].set_title('Spearman Correlation (Dev)')
axes[0].set_ylabel('Spearman ρ')
axes[0].set_ylim(-0.2, 1.0)
for i, v in enumerate(spearman_vals):
    axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=10)

axes[1].bar(models, acc_vals, color=colors, edgecolor='white')
axes[1].set_title('Accuracy Within Std Dev (Dev)')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0, 1.0)
for i, v in enumerate(acc_vals):
    axes[1].text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('baselines_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: baselines_comparison.png')

## ✅ Notebook 1 Complete
Files produced:
- `eda_score_distribution.png`
- `eda_stdev_vs_score.png`
- `baselines_results.json`
- `baselines_comparison.png`

Next: Run **notebook2_feature_engineering.ipynb**